In [5]:
!pip install littlelearn --no-deps
import littlelearn as ll
from littlelearn import DeepLearning as dl
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression

## Multi Linear Layer Regression
aku sebut itu karena kita akan mencoba memungkinkan sebuat model
melakukan pemahaman terhadap suatu data namun tidak secara satu layer melainkan melakukan transformasi berlapis dan inilah dia bagaimana mekanisme Neural Network bekerja pada sebelum menemui activation nanti

In [24]:

# ingat bahwa rumus Linear layer pada NN adalah f (x) = x dot w + b
#sebenarnya kalian hanya tinggal seperti ini saja
sample = ll.rand(5)
layers = dl.layers.Linear(5,1)
print(layers(sample))
# kita akan pakai API disini namun juga tetap masih implementasi manual dengan cara membuat layers sendiri
# kita akan pakai abstraction class bernama Component sebagai penanda bahwa ini bagian dari component Model
# yang akan dipakai pada model lainnya dan seterusnya
class Linear (dl.layers.Component) :
  def __init__(self,input_dim,output_dim,bias = False) :
    super().__init__()
    # disini kita akan coba tanpa mekanisme initial sama sekali
    self.w = dl.layers.Parameter(ll.uniform(low=-0.01 , high = 0.01, shape=(input_dim,output_dim),max_random_seed=50))
    self.b = dl.layers.Parameter(ll.zeros(shape=(output_dim,)))

  def forwardpass(self,x) :
    return ll.matmul(x,self.w) + self.b

Tensor(shape=(1, 1), dtype=<class 'jax.numpy.float32'>, device=cpu, requires_grad=False)
[[-0.5762578]]
Backwardpass class : None


##  disini kita akan membuktikan bagaimana aliran training bekerja antar layer ke layer tanpa container sama sekali. sebenarnya kalian dapat pakai Component class untuk itu, namun kita mau lihat bagaiamana aliran tensor bekerja disini.

In [27]:
x_train,y_train = make_regression()
y_train = y_train.reshape(-1,1)
layer1 = Linear(x_train.shape[-1],128)
layer2 = Linear(128,256)
layer3 = Linear(256,1)
param = []
for l in [layer1,layer2,layer3] :
  for p in l.parameter() :
    param.append(p)
  l.train(True)
optim = dl.optimizers.Adam(param,lr=0.001)

name : Linear status : active || requires_grad : True
name : Linear status : active || requires_grad : True
name : Linear status : active || requires_grad : True


In [28]:
#kita ingin melihat bagaimana perubahan dimensi antar input ke layer dan layer ke layer hingga output saat real training
for epoch in range(10) :
  x = layer1(x_train)
  print(f"dimensi input layer 1 : {x.shape}")
  x = layer2(x)
  print(f"dimensi input layer 2 : {x.shape}")
  x = layer3(x)
  print(f"dimensi input layer 3 : {x.shape}")
  loss = dl.loss.mse_loss(y_train,x)
  loss.backwardpass()
  optim.step()
  loss.reset_grad()
  print(loss)


dimensi input layer 1 : (100, 128)
dimensi input layer 2 : (100, 256)
dimensi input layer 3 : (100, 1)
Tensor(shape=(), dtype=<class 'jax.numpy.float32'>, device=cpu, requires_grad=True)
42764.15234375
Backwardpass class : method
dimensi input layer 1 : (100, 128)
dimensi input layer 2 : (100, 256)
dimensi input layer 3 : (100, 1)
Tensor(shape=(), dtype=<class 'jax.numpy.float32'>, device=cpu, requires_grad=True)
42763.5078125
Backwardpass class : method
dimensi input layer 1 : (100, 128)
dimensi input layer 2 : (100, 256)
dimensi input layer 3 : (100, 1)
Tensor(shape=(), dtype=<class 'jax.numpy.float32'>, device=cpu, requires_grad=True)
42762.01953125
Backwardpass class : method
dimensi input layer 1 : (100, 128)
dimensi input layer 2 : (100, 256)
dimensi input layer 3 : (100, 1)
Tensor(shape=(), dtype=<class 'jax.numpy.float32'>, device=cpu, requires_grad=True)
42758.828125
Backwardpass class : method
dimensi input layer 1 : (100, 128)
dimensi input layer 2 : (100, 256)
dimensi input

In [33]:
#sekarang kita akan coba bagaimana membuat model lansung dengan Component
class Model (dl.layers.Component) :
  def __init__(self,input_dim) :
    self.layer1 = Linear(input_dim,128)
    self.layer2 = Linear(128,256)
    self.layer3 = Linear(256,1)

  def forwardpass(self,x) :
    x = self.layer1(x)
    x = self.layer2(x)
    return self.layer3(x)

In [35]:
model = Model(x_train.shape[-1])
model.train(True)
optim = dl.optimizers.Adam(model.parameter(),lr=0.01)
y_train = y_train
for epoch in range(10) :
  x = model(x_train)
  loss = dl.loss.mse_loss(y_train,x)
  loss.backwardpass()
  optim.step()
  loss.reset_grad()
  print(loss.tensor)

name : Model status : active || requires_grad : True
42764.176
42754.48
42523.562
41953.11
40391.793
36975.4
31216.475
23770.92
17969.727
17900.654
